# EcoGuard-ML Core: 01 Dataset Creation Pipeline

This notebook handles the generation of 10,000 synthetic wildlife monitoring records. Each record mimics real-world sensor streams from an East African wildlife reserve. It models temporal, environmental, geographic, and wildlife variables, culminating in a rule-based `poaching_incident` label that reflects realistic threat dynamics.

### Section Breakdown:
1. **Imports & Setup:** Loading standard scientific Python packages and configuring random seeds for reproducibility.
2. **Geographical Coordinates & Zones:** Simulating coordinates in a specific reserve bounding box, mapping them into grid zones, and generating terrain elevations.
3. **Geodesic Distance Metrics:** Calculating distances from events to simulated roads, water bodies, and ranger stations using the Haversine formula.
4. **Temporal & Climatic Modeling (with Missing Values):** Simulating diurnal cycles for temperature, inverse relationships for humidity, and seasonal rainfall, with **exactly 155 missing values** injected in `rainfall` for EDA purposes.
5. **Wildlife Density Score & Species Simulation:** Modelling animal concentration scores (scaled `[0, 100]`) and identifying the primary species present in the event block (`Elephant`, `Rhino`, `Lion`, `Buffalo`, `Zebra`, `None Detected`).
6. **Historical Hotspots & Acoustic Telemetry:** Generating zone-specific historical crime counts and normalized live acoustic risk indexes.
7. **Rule-Based Threat Score & Poaching Labels:** Generating the classification target label using a deterministic operational risk function involving species value, historical counts, and acoustic alarms.
8. **Data Assembly & CSV Export:** Creating a DataFrame matching the schema exactly and saving it as `master_dataset.csv`.
9. **Diagnostics & Validation:** Verifying class balance, spatial/temporal distributions, missing value counts, and correlation logic.

## Section 1: Imports & Setup

We import standard packages (`numpy`, `pandas`, `math`, `os`, `uuid`) and seed the random generators so that this synthetic dataset yields identical values across runs.

In [ ]:
import numpy as np
import pandas as pd
import math
import os
import uuid

# Set random seeds for absolute reproducibility
np.random.seed(42)

# Dataset configuration
NUM_RECORDS = 10000

# Geographic boundary for the simulated reserve (e.g., East African Serengeti region)
LAT_MIN, LAT_MAX = -2.5, -1.5
LON_MIN, LON_MAX = 34.5, 35.5

print(f"Setup completed. Set to generate {NUM_RECORDS} records.")

## Section 2: Geographic Coordinates, Zones, & Elevation

Here, we:
1. Generate uniform latitude and longitude points inside our reserve bounding box.
2. Bin the reserve into a 5x5 grid map (25 zones from `ZONE_A01` to `ZONE_E05`).
3. Simulate elevation using a spatiotemporal topographic function where the center region is higher (hills/mountains) and add Gaussian noise.

In [ ]:
# 1. Non-uniform Coordinates (clustered hotspots)
np.random.seed(42)
latitudes = []
longitudes = []
# Define 3 hotspots representing concentration zones
hotspots = [
    (-1.8, 34.8),  # water/river area
    (-2.3, 35.1),  # road/forest border
    (-2.0, 35.0)   # center region
]
for _ in range(NUM_RECORDS):
    # 60% probability of being near a hotspot, 40% uniform
    if np.random.random() < 0.6:
        lat_h, lon_h = hotspots[np.random.randint(len(hotspots))]
        # Gaussian cluster with std = 0.05
        latitudes.append(np.random.normal(lat_h, 0.05))
        longitudes.append(np.random.normal(lon_h, 0.05))
    else:
        latitudes.append(np.random.uniform(LAT_MIN, LAT_MAX))
        longitudes.append(np.random.uniform(LON_MIN, LON_MAX))
latitudes = np.array(latitudes)
longitudes = np.array(longitudes)
# Clamp coordinates within reserve bounding box
latitudes = np.clip(latitudes, LAT_MIN, LAT_MAX)
longitudes = np.clip(longitudes, LON_MIN, LON_MAX)

# 2. Grid Zone Mapping
def coordinate_to_zone(lat, lon):
    lat_idx = int((lat - LAT_MIN) / (LAT_MAX - LAT_MIN) * 5)
    lat_idx = min(max(lat_idx, 0), 4)
    lat_letter = ['A', 'B', 'C', 'D', 'E'][lat_idx]
    lon_idx = int((lon - LON_MIN) / (LON_MAX - LON_MIN) * 5) + 1
    lon_idx = min(max(lon_idx, 1), 5)
    return f"ZONE_{lat_letter}{lon_idx:02d}"

zones = [coordinate_to_zone(lat, lon) for lat, lon in zip(latitudes, longitudes)]

# 3. Elevation Simulation (Hilly terrain centered in the park)
elevation_base = 1200  # meters base
dist_from_center = np.sqrt((latitudes - (-2.0))**2 + (longitudes - 35.0)**2)
elevations = elevation_base + 400 * np.exp(-dist_from_center / 0.4) + np.random.normal(0, 30, NUM_RECORDS)
elevations = np.round(elevations, 1)

print("Geographic features simulated with clustered spatial hotspots. Example zone:", zones[0], "elevation:", elevations[0], "m")


## Section 3: Geodesic Distance Metrics

In wildlife threat modeling, proximity to infrastructure dictates both animal and human behavior. We:
1. Define the locations of key linear structures (roads) and nodes (rivers/lakes, ranger stations).
2. Implement the **Haversine Formula** to compute correct physical distances in meters from coordinates on the Earth's sphere.
3. Compute the distance to the nearest road, water body, and ranger station for each event.

In [ ]:
# Haversine formula for distance in meters
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth's radius in meters
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    d_phi = np.radians(lat2 - lat1)
    d_lambda = np.radians(lon2 - lon1)
    
    a = np.sin(d_phi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(d_lambda / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c

# Define infrastructure nodes
# 3 Ranger Outposts
RANGER_STATIONS = [
    {"lat": -2.0, "lon": 35.0, "name": "Central Station"},
    {"lat": -1.6, "lon": 35.4, "name": "Northeast Station"},
    {"lat": -2.4, "lon": 34.6, "name": "Southwest Station"}
]

# 3 Water Bodies (lakes/river bends)
WATER_BODIES = [
    {"lat": -1.8, "lon": 34.8},
    {"lat": -2.2, "lon": 35.2},
    {"lat": -2.1, "lon": 34.6}
]

# Roads (simulated as linear paths by sampling points along the lines)
road_points = []
for l in np.linspace(34.5, 35.5, 50):
    road_points.append((-2.3, l))
for la in np.linspace(-2.5, -1.5, 50):
    road_points.append((la, 34.7))

# Compute distances for all records
dist_to_ranger = []
dist_to_water = []
dist_to_road = []

for lat, lon in zip(latitudes, longitudes):
    # Ranger station distance
    ranger_dists = [haversine_distance(lat, lon, rs["lat"], rs["lon"]) for rs in RANGER_STATIONS]
    dist_to_ranger.append(min(ranger_dists))
    
    # Water body distance
    water_dists = [haversine_distance(lat, lon, wb["lat"], wb["lon"]) for wb in WATER_BODIES]
    dist_to_water.append(min(water_dists))
    
    # Road distance
    road_dists = [haversine_distance(lat, lon, r_lat, r_lon) for r_lat, r_lon in road_points]
    dist_to_road.append(min(road_dists))

dist_to_ranger = np.round(dist_to_ranger, 1)
dist_to_water = np.round(dist_to_water, 1)
dist_to_road = np.round(dist_to_road, 1)

print("Distance metrics computed. Examples (ranger, water, road in meters):")
print(dist_to_ranger[0], dist_to_water[0], dist_to_road[0])

## Section 4: Temporal, Climatic Variables & Missing Values

We simulate a realistic climate environment with **exactly 155 missing values** injected into `rainfall`:
1. **Month & Hour:** Uniformly distributed.
2. **Season:** Derived from months (Wet, Dry, Short Wet, Short Dry).
3. **Temperature:** Cosine-wave diurnal temperature cycle peaking at 14:00.
4. **Humidity:** Modeled to run inversely to temperature, boosted by season.
5. **Rainfall:** Simulated using Gamma distributions, with 155 random entries replaced by `np.nan` to mimic weather sensor drops.

In [ ]:
import datetime

# Generate chronological sequential timestamps over 2025
np.random.seed(42)
start_date = datetime.datetime(2025, 1, 1, 0, 0, 0)
mean_step = 31536000.0 / NUM_RECORDS  # average step size in seconds
current_time = start_date
timestamps = []
for _ in range(NUM_RECORDS):
    step = np.random.exponential(mean_step)
    current_time += datetime.timedelta(seconds=step)
    timestamps.append(current_time)
    
timestamps = sorted(timestamps)

months = np.array([ts.month for ts in timestamps])
hours = np.array([ts.hour for ts in timestamps])

seasons = []
for m in months:
    if m in [3, 4, 5]:
        seasons.append("Wet")
    elif m in [6, 7, 8, 9, 10]:
        seasons.append("Dry")
    elif m in [11, 12]:
        seasons.append("Short Wet")
    else:
        seasons.append("Short Dry")
        
temperatures = []
humidities = []
rainfalls = []

for hr, m, seas in zip(hours, months, seasons):
    # Base temperature by season
    if seas == "Dry":
        base_t = 26.0
    elif seas == "Wet":
        base_t = 21.0
    elif seas == "Short Dry":
        base_t = 24.0
    else:
        base_t = 22.5
    
    # Diurnal variation: cosine-wave diurnal temperature cycle peaking at 14:00
    diurnal_effect = 5.5 * math.cos((hr - 14.0) * math.pi / 12.0)
    temp = base_t + diurnal_effect + np.random.normal(0, 1.2)
    temperatures.append(round(temp, 1))
    
    # Humidity: inversely correlated to diurnal temp, boosted by season
    if seas in ["Wet", "Short Wet"]:
        base_h = 80.0
    else:
        base_h = 50.0
    
    humidity = base_h - 1.8 * diurnal_effect + np.random.normal(0, 3.0)
    humidity = min(max(humidity, 15.0), 100.0)
    humidities.append(round(humidity, 1))
    
    # Rainfall: Gamma distribution modeled by season probability
    if seas == "Wet":
        rain_prob = 0.55
        amount = np.random.gamma(shape=2.5, scale=8.0) if np.random.random() < rain_prob else 0.0
    elif seas == "Short Wet":
        rain_prob = 0.35
        amount = np.random.gamma(shape=2.0, scale=6.0) if np.random.random() < rain_prob else 0.0
    elif seas == "Short Dry":
        rain_prob = 0.15
        amount = np.random.gamma(shape=1.5, scale=4.0) if np.random.random() < rain_prob else 0.0
    else:
        rain_prob = 0.05
        amount = np.random.gamma(shape=1.0, scale=3.0) if np.random.random() < rain_prob else 0.0
    rainfalls.append(round(amount, 1))
    
# Inject exactly 155 missing rainfall values for EDA validation
null_indices = np.random.choice(NUM_RECORDS, 155, replace=False)
for idx in null_indices:
    rainfalls[idx] = np.nan
    
print("Temporal and meteorological parameters simulated sequentially. Mean temp:", round(np.mean(temperatures), 1))


## Section 5: Wildlife Activity (Animal Density Score) & Species Simulation

We model wildlife concentration and species presence:
1. **`animal_density_score`**: Scaled to `[0, 100]` for simplified interpretation. It models animals gathering near water and avoiding roads, modified by dry season constraints.
2. **`species`**: `["Elephant", "Rhino", "Lion", "Buffalo", "Zebra", "None Detected"]`. Low density score (< 18) defaults to `"None Detected"` (using a specific string to avoid default Pandas NaN parsing); others map probabilistically based on spatial distance metrics.

In [ ]:
animal_density_scores = []
raw_densities = []  # save float density for species distribution logic

for d_water, d_road, seas in zip(dist_to_water, dist_to_road, seasons):
    decay_rate = 1200.0 if seas == "Dry" else 2500.0
    water_pull = math.exp(-d_water / decay_rate)
    road_avoidance = 1.0 - math.exp(-d_road / 2000.0)
    base_density = 0.6 * water_pull + 0.25 * road_avoidance + np.random.uniform(0.05, 0.15)
    density = min(max(base_density, 0.0), 1.0)
    raw_densities.append(density)
    
    # Scale to [0, 100] as requested for interpretation ease
    density_score = int(round(density * 100))
    animal_density_scores.append(density_score)

species_list = []
for density, d_water, d_road in zip(raw_densities, dist_to_water, dist_to_road):
    if density < 0.18:
        species_list.append("None Detected")
    else:
        probs = {
            "Elephant": 0.25 if d_water < 2000 else 0.10,
            "Rhino": 0.12 if (d_road > 5000 and d_water < 3000) else 0.02,
            "Lion": 0.15 if density > 0.6 else 0.08,
            "Buffalo": 0.20,
            "Zebra": 0.35
        }
        total_p = sum(probs.values())
        keys = list(probs.keys())
        p_vals = [probs[k]/total_p for k in keys]
        species_list.append(np.random.choice(keys, p=p_vals))

print("Wildlife and species simulation completed. Mean animal density score:", round(np.mean(animal_density_scores), 1))

## Section 6: Historical Crime Hotspots & Live Acoustic Risks

To enrich our threat parameters, we simulate:
1. **`historical_incident_count`**: We establish spatial hotspots in the 25 zones. Northwest zones (near park boundaries and roads) are assigned high counts (15-35 incidents), while secure zones near central ranger stations have low counts (0-8).
2. **`acoustic_risk`**: Normalized index representing warnings from local listening nodes. Acoustic anomalies spike close to roads (vehicle noises), in remote target areas with high animal density (chainsaw/human sounds), and slightly during nocturnal periods.

In [ ]:
unique_zones = sorted(list(set(zones)))
zone_history = {}

for z in unique_zones:
    row_val = ord(z[5]) - ord('A')  # 0 to 4
    col_val = int(z[6:]) - 1         # 0 to 4
    
    if row_val <= 1 and col_val <= 1:
        zone_history[z] = np.random.randint(15, 36)
    elif row_val >= 3 and col_val >= 3:
        zone_history[z] = np.random.randint(8, 18)
    else:
        zone_history[z] = np.random.randint(0, 8)

historical_incidents = [zone_history[z] for z in zones]

acoustic_risks = []
for d_road, density, hr in zip(dist_to_road, raw_densities, hours):
    road_noise = math.exp(-d_road / 2500.0)
    remote_risk = density * (1.0 - math.exp(-d_road / 5000.0))
    night_effect = 0.15 if (hr >= 20 or hr <= 4) else 0.0
    
    score = 0.35 * road_noise + 0.35 * remote_risk + night_effect + np.random.uniform(0.0, 0.15)
    score = min(max(score, 0.0), 1.0)
    acoustic_risks.append(round(score, 2))

print("Spatial hotspots and acoustic tracking simulated. Mean acoustic risk:", round(np.mean(acoustic_risks), 2))

## Section 7: Updated Rule-Based Threat Score & Poaching Labels

We modify the rule-based risk logic to weave in our new features, utilizing the normalized `animal_density_score / 100` as the animal risk factor. The composite score is thresholded to preserve our target ~7.5% poaching incident rate.

In [ ]:
species_weights = {
    "Elephant": 1.0,
    "Rhino": 1.0,
    "Lion": 0.6,
    "Buffalo": 0.4,
    "Zebra": 0.1,
    "None Detected": 0.0
}

threat_probs = []
for d_road, d_ranger, d_water, sp, h_count, a_risk, hr, d_score, rain in zip(
    dist_to_road, dist_to_ranger, dist_to_water, species_list, historical_incidents, acoustic_risks, hours, animal_density_scores, rainfalls
):
    # Stochastic risk factors
    # Time of day: poaching is significantly more likely during the cover of night
    p_nocturnal = 0.40 if (hr >= 22 or hr <= 4) else 0.05
    
    # Weather: heavy rain renders dirt roads and off-road trails impassable, suppressing threat
    actual_rain = 0.0 if np.isnan(rain) else rain
    p_rain_modifier = 0.15 if actual_rain > 15.0 else 1.0
    
    # Road accessibility: proximity to local access roads is essential for transport
    road_dist_km = d_road / 1000.0
    p_road_modifier = math.exp(-road_dist_km / 3.0)  # decays with distance
    
    # Ranger station proximity: proximity to operational outposts acts as a strong deterrent
    ranger_dist_km = d_ranger / 1000.0
    p_ranger_modifier = 1.0 - 0.85 * math.exp(-ranger_dist_km / 5.0)
    
    # Wildlife density & high-value target presence
    p_wildlife_modifier = 0.2 + 0.8 * (d_score / 100.0)
    sp_val = species_weights[sp]
    
    # Historical crime footprint
    p_history_modifier = 0.3 + 0.7 * (h_count / 35.0)
    
    # Acoustic threat risk (immediate indicator)
    p_acoustic_modifier = a_risk
    
    # Combine stochastically
    p_base = (
        p_nocturnal *
        p_rain_modifier *
        p_road_modifier *
        p_ranger_modifier *
        (0.3 * p_wildlife_modifier * sp_val + 0.3 * p_history_modifier + 0.4 * p_acoustic_modifier)
    )
    
    threat_probs.append(p_base)
    
threat_probs = np.array(threat_probs)

# Normalize threat probabilities so that the expected value (mean probability)
# matches our target poaching incident rate of exactly 7.50%
target_rate = 0.075
current_mean = np.mean(threat_probs)
scaling_factor = target_rate / current_mean

scaled_probs = threat_probs * scaling_factor
# Clamp probabilities to [0.005, 0.95]
scaled_probs = np.clip(scaled_probs, 0.005, 0.95)

# Stochastic incident generation using Bernoulli trials
np.random.seed(42)
poaching_incidents = np.random.binomial(1, scaled_probs)

print(f"Stochastic threat calculation complete. Average probability: {np.mean(scaled_probs):.4f}")
print(f"Incident count: {sum(poaching_incidents)} positive logs ({np.mean(poaching_incidents)*100:.2f}%).")


## Section 8: Data Assembly & CSV Export

We compile all features into a Pandas DataFrame in the exact order specified by the updated [docs/dataset_schema.md](file:///c:/Users/ADMIN/OneDrive/Desktop/ecogaurd_ml/docs/dataset_schema.md) file, using the renamed `animal_density_score` variable.

In [ ]:
event_ids = [f"evt_{uuid.uuid4().hex[:8]}" for _ in range(NUM_RECORDS)]

df_dict = {
    "event_id": event_ids,
    "zone_id": zones,
    "latitude": np.round(latitudes, 5),
    "longitude": np.round(longitudes, 5),
    "temperature": temperatures,
    "humidity": humidities,
    "rainfall": rainfalls,
    "animal_density_score": animal_density_scores,
    "distance_to_road": dist_to_road,
    "distance_to_water": dist_to_water,
    "distance_to_ranger_station": dist_to_ranger,
    "elevation": elevations,
    "hour": hours,
    "month": months,
    "season": seasons,
    "species": species_list,
    "historical_incident_count": historical_incidents,
    "acoustic_risk": acoustic_risks,
    "poaching_incident": poaching_incidents
}

master_df = pd.DataFrame(df_dict)

# Check if parent data/raw exists, otherwise use fallback
raw_dir = "../data/raw"
if not os.path.exists(raw_dir):
    raw_dir = "data/raw"
os.makedirs(raw_dir, exist_ok=True)
csv_path = os.path.join(raw_dir, "master_dataset.csv")
master_df.to_csv(csv_path, index=False)

print(f"Master dataset successfully compiled and exported to: {csv_path}")


## Section 9: Diagnostics & Validation

We verify that our dataset complies with the target parameters and check for missing value statistics and density ranges.

In [ ]:
print("=== Dataset Diagnostics ===\n")
print(f"Dataset Shape: {master_df.shape}")
print(f"Incident Class Balance:\n{master_df['poaching_incident'].value_counts(normalize=True) * 100}\n")

print(f"Total Missing Values across DataFrame: {master_df.isnull().sum().sum()}")
print(f"Missing Values by Column:\n{master_df.isnull().sum()}\n")

print(f"animal_density_score limits: Min={master_df['animal_density_score'].min()}, Max={master_df['animal_density_score'].max()}")

print("\n=== Feature Means by Class (Poaching vs. Safe) ===")
means_by_class = master_df.groupby("poaching_incident")[
    ["distance_to_road", "distance_to_ranger_station", "distance_to_water", "animal_density_score", "historical_incident_count", "acoustic_risk"]
].mean().T
means_by_class.columns = ["No Incident (0)", "Incident (1)"]
print(means_by_class.to_string())

print("\n=== Head of Master Dataset ===")
print(master_df.head().to_string())